# Notebook 07 — CP02: Analysis and Writeup

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 7 of 12  
**Time estimate:** 75 minutes

> This notebook produces the final comparison, the statistical test,
> the written abstract, and the methods section for CP02.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# Reference AUROC values from NB05–NB06 (pre-computed)
lr_auroc  = np.array([0.891, 0.883, 0.888, 0.885, 0.887])
svm_auroc = np.array([0.912, 0.908, 0.910, 0.905, 0.914])
lr_auprc  = np.array([0.889, 0.881, 0.885, 0.882, 0.886])
svm_auprc = np.array([0.911, 0.905, 0.908, 0.903, 0.912])
cnn_auroc = np.array([0.986, 0.981, 0.983, 0.979, 0.984])
cnn_auprc = np.array([0.985, 0.979, 0.982, 0.977, 0.983])

# Paired Wilcoxon: CNN vs best baseline (SVM)
stat_auroc, pval_auroc = stats.wilcoxon(cnn_auroc, svm_auroc, alternative='greater')
stat_auprc, pval_auprc = stats.wilcoxon(cnn_auprc, svm_auprc, alternative='greater')
diff_pp  = (cnn_auroc - svm_auroc).mean() * 100
ci_lo, ci_hi = np.percentile(
    [rng.choice(cnn_auroc-svm_auroc, 5, replace=True).mean() for _ in range(5000)],
    [2.5, 97.5])

print('=== CP02: Full comparison table ===')
print(f'{"Method":<15} {"AUROC mean":>12} {"AUROC SD":>10} {"AUPRC mean":>12} {"AUPRC SD":>10}')
print('-'*62)
for name, au, ap in [("k-mer+LR",lr_auroc,lr_auprc),("k-mer+SVM",svm_auroc,svm_auprc),("CNN",cnn_auroc,cnn_auprc)]:
    print(f'{name:<15} {au.mean():>12.3f} {au.std():>10.3f} {ap.mean():>12.3f} {ap.std():>10.3f}')
print(f'\nCNN vs SVM (AUROC, Wilcoxon signed-rank, n=5):')
print(f'  Improvement: {diff_pp:.1f} pp (95% CI: {ci_lo*100:.1f}–{ci_hi*100:.1f} pp)')
print(f'  p = {pval_auroc:.4f}', '  ***' if pval_auroc < 0.001 else '  *' if pval_auroc < 0.05 else '  ns')

In [ ]:
# Publication rcParams
plt.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300,
})

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Panel A: Full comparison (violin + dots)
ax = axes[0]
methods_data = [lr_auroc, svm_auroc, cnn_auroc]
colors_c = ['#59a14f', '#f28e2b', '#4e79a7']
parts = ax.violinplot(methods_data, positions=[1,2,3], widths=0.6, showmeans=True)
for i, (pc, color) in enumerate(zip(parts['bodies'], colors_c)):
    pc.set_facecolor(color); pc.set_alpha(0.7)
for k in ['cmins','cmaxes','cbars','cmeans']:
    parts[k].set_color('black'); parts[k].set_lw(1.5)
for i, (data, color, pos) in enumerate(zip(methods_data, colors_c, [1,2,3])):
    ax.scatter(np.full(5, pos) + rng.uniform(-0.05, 0.05, 5), data, color='black', s=30, zorder=3)
ax.set_xticks([1,2,3]); ax.set_xticklabels(['k-mer\n+LR', 'k-mer\n+SVM', 'CNN'])
ax.set_ylabel('AUROC (5-fold CV)')
ax.set_ylim(0.82, 1.02)
ax.set_title('A. Method comparison\n(5-fold CV; mean ± SD)')
ax.text(2.5, cnn_auroc.mean() + 0.005, '***', ha='center', fontsize=12)
ax.plot([2, 3], [0.98, 0.98], 'k-', lw=1)
ax.text(-0.12, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel B: Improvement distribution
ax = axes[1]
improve_pp = (cnn_auroc - svm_auroc) * 100
ax.bar(range(5), improve_pp, color='#4e79a7', edgecolor='black', alpha=0.8)
ax.axhline(improve_pp.mean(), color='tomato', ls='--', lw=2, label=f'Mean={improve_pp.mean():.1f} pp')
ax.fill_between([-0.5, 4.5], ci_lo*100, ci_hi*100, alpha=0.2, color='tomato', label='95% CI')
ax.set_xticks(range(5)); ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_ylabel('AUROC improvement over SVM (pp)')
ax.set_title('B. CNN improvement per fold\n(CNN − k-mer+SVM)')
ax.legend(fontsize=8)
ax.text(-0.12, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold')

# Panel C: Abstract and methods preview
ax = axes[2]
ax.axis('off')
abstract_text = (
    "Abstract (6-sentence template):\n\n"
    "S1 [Context]: Transcription factor (TF) binding site prediction from "
    "DNA sequence is fundamental to understanding gene regulation.\n\n"
    "S2 [Gap]: Existing position weight matrix methods treat nucleotide "
    "positions as independent, failing to model cooperative binding.\n\n"
    "S3 [Approach]: Here we present TFBindingCNN, a convolutional neural "
    "network trained on one-hot encoded sequences from the GATAAG motif benchmark.\n\n"
    f"S4 [Result]: TFBindingCNN achieves AUROC {cnn_auroc.mean():.3f} ± {cnn_auroc.std():.3f}, "
    f"outperforming the k-mer+SVM baseline by {diff_pp:.1f} pp "
    f"(Wilcoxon, p={pval_auroc:.4f}, n=5 folds).\n\n"
    "S5 [Mechanism]: Learned filters recover the GATAAG consensus motif, "
    "demonstrating that position-specific features drive the improvement.\n\n"
    "S6 [Significance]: Code available at github.com/Vinoth-ai-20/computational-biology."
)
ax.text(0.03, 0.97, abstract_text, transform=ax.transAxes, fontsize=7.5, va='top',
          bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='grey'))
ax.set_title('C. Draft abstract (6-sentence template)')
ax.text(-0.12, 1.05, 'C', transform=ax.transAxes, fontsize=12, fontweight='bold')

plt.suptitle('Module 20 CP02: Analysis and Writeup', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp02_writeup.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Write the 400-word methods section for CP02 using the completeness checklist
> from Module 18 NB04. Run `audit_methods_text()` on your draft.
> What is the most important missing item?]*

---
*Next: `08_cp03_model_implementation.ipynb`*